# 💾 Response Caching

**Speed up your LLM API with caching**

---

## 📋 Overview

**What you'll learn:**
- Why cache LLM responses
- In-memory caching
- Redis caching
- Cache invalidation
- TTL strategies

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
import time
import hashlib
import json
from typing import Optional, Dict, Any

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Cache LLM Responses?

### Without Caching:

```
User: "What is 2+2?"
  ↓
API calls OpenAI ($0.002)
  ↓
Wait 1.5s
  ↓
Return: "4"

User: "What is 2+2?"  (same question!)
  ↓
API calls OpenAI again ($0.002)
  ↓
Wait 1.5s
  ↓
Return: "4"

❌ Problems:
- Wasted money ($0.004 for same question)
- Slow (3s total)
- Unnecessary API calls
```

### With Caching:

```
User: "What is 2+2?"
  ↓
Check cache: MISS
  ↓
Call OpenAI ($0.002)
  ↓
Store in cache
  ↓
Return: "4" (1.5s)

User: "What is 2+2?"  (same question!)
  ↓
Check cache: HIT ✅
  ↓
Return: "4" (0.001s)

✅ Benefits:
- Save money (50% cost reduction)
- Faster (1000x for cached responses)
- Reduce API load
```

### When to Cache:

**Good candidates:**
- Repeated questions
- Static prompts
- Popular queries
- Expensive operations

**Don't cache:**
- User-specific data
- Real-time information
- Randomized outputs
- Sensitive data

## 💾 Simple In-Memory Cache

In [ ]:
import hashlib
import json
from typing import Dict, Optional

class SimpleCache:
    """Simple in-memory cache."""
    
    def __init__(self):
        self.cache: Dict[str, str] = {}
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, prompt: str, model: str) -> str:
        """Create cache key from prompt and model."""
        data = json.dumps({"prompt": prompt, "model": model}, sort_keys=True)
        return hashlib.md5(data.encode()).hexdigest()
    
    def get(self, prompt: str, model: str) -> Optional[str]:
        """Get cached response."""
        key = self._make_key(prompt, model)
        
        if key in self.cache:
            self.hits += 1
            print(f"  ✅ Cache HIT")
            return self.cache[key]
        
        self.misses += 1
        print(f"  ❌ Cache MISS")
        return None
    
    def set(self, prompt: str, model: str, response: str):
        """Store response in cache."""
        key = self._make_key(prompt, model)
        self.cache[key] = response
        print(f"  💾 Cached response")
    
    def get_stats(self) -> Dict:
        """Get cache statistics."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1f}%",
            "size": len(self.cache)
        }

# Example usage
cache = SimpleCache()

def get_completion_cached(prompt: str, model: str = "gpt-3.5-turbo") -> str:
    """Get LLM completion with caching."""
    
    # Check cache first
    cached = cache.get(prompt, model)
    if cached:
        return cached
    
    # Cache miss - call API
    print(f"  🌐 Calling OpenAI API...")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = response.choices[0].message.content
    
    # Store in cache
    cache.set(prompt, model, result)
    
    return result

# Test caching
print("\n💾 Testing Simple Cache\n")

print("Request 1:")
response1 = get_completion_cached("What is 2+2?")
print(f"  Response: {response1[:50]}...\n")

print("Request 2 (same prompt):")
response2 = get_completion_cached("What is 2+2?")
print(f"  Response: {response2[:50]}...\n")

print("Cache stats:")
stats = cache.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

## ⏰ TTL (Time-To-Live) Cache

In [ ]:
import time
from typing import Dict, Optional, Tuple

class TTLCache:
    """Cache with time-to-live expiration."""
    
    def __init__(self, ttl_seconds: int = 3600):
        self.ttl_seconds = ttl_seconds
        self.cache: Dict[str, Tuple[str, float]] = {}
    
    def _make_key(self, prompt: str, model: str) -> str:
        """Create cache key."""
        data = json.dumps({"prompt": prompt, "model": model}, sort_keys=True)
        return hashlib.md5(data.encode()).hexdigest()
    
    def _is_expired(self, timestamp: float) -> bool:
        """Check if cache entry is expired."""
        return time.time() - timestamp > self.ttl_seconds
    
    def get(self, prompt: str, model: str) -> Optional[str]:
        """Get cached response if not expired."""
        key = self._make_key(prompt, model)
        
        if key in self.cache:
            response, timestamp = self.cache[key]
            
            if not self._is_expired(timestamp):
                age = time.time() - timestamp
                print(f"  ✅ Cache HIT (age: {age:.1f}s)")
                return response
            else:
                # Expired - remove from cache
                del self.cache[key]
                print(f"  ⏰ Cache EXPIRED")
        
        print(f"  ❌ Cache MISS")
        return None
    
    def set(self, prompt: str, model: str, response: str):
        """Store response with timestamp."""
        key = self._make_key(prompt, model)
        self.cache[key] = (response, time.time())
        print(f"  💾 Cached (TTL: {self.ttl_seconds}s)")
    
    def cleanup_expired(self) -> int:
        """Remove all expired entries."""
        expired_keys = [
            key for key, (_, timestamp) in self.cache.items()
            if self._is_expired(timestamp)
        ]
        
        for key in expired_keys:
            del self.cache[key]
        
        return len(expired_keys)

# Example with different TTLs
print("\n⏰ TTL Cache Examples\n")

# Short TTL for dynamic content
short_cache = TTLCache(ttl_seconds=60)  # 1 minute
print("Short TTL (1 minute) - for dynamic content")

# Long TTL for static content
long_cache = TTLCache(ttl_seconds=86400)  # 24 hours
print("Long TTL (24 hours) - for static content")

print("\n💡 TTL Strategy:")
print("  - Static prompts: 24 hours")
print("  - News/weather: 5-15 minutes")
print("  - User queries: 1 hour")
print("  - Expensive operations: 6-12 hours")

## 🔴 Redis Cache

In [ ]:
print("""
# Production caching with Redis

import redis
import json
import hashlib
from typing import Optional

class RedisCache:
    \"\"\"Redis-based cache for production.\"\"\" 
    
    def __init__(self, host='localhost', port=6379, ttl=3600):
        self.redis = redis.Redis(
            host=host,
            port=port,
            decode_responses=True
        )
        self.ttl = ttl
    
    def _make_key(self, prompt: str, model: str) -> str:
        \"\"\"Create cache key.\"\"\" 
        data = json.dumps({"prompt": prompt, "model": model}, sort_keys=True)
        hash_key = hashlib.md5(data.encode()).hexdigest()
        return f"llm_cache:{hash_key}"
    
    def get(self, prompt: str, model: str) -> Optional[str]:
        \"\"\"Get cached response.\"\"\" 
        key = self._make_key(prompt, model)
        
        value = self.redis.get(key)
        
        if value:
            # Track cache hit
            self.redis.incr("cache:hits")
            return value
        
        # Track cache miss
        self.redis.incr("cache:misses")
        return None
    
    def set(self, prompt: str, model: str, response: str):
        \"\"\"Store response with TTL.\"\"\" 
        key = self._make_key(prompt, model)
        
        # Set with expiration
        self.redis.setex(key, self.ttl, response)
    
    def get_stats(self) -> dict:
        \"\"\"Get cache statistics.\"\"\" 
        hits = int(self.redis.get("cache:hits") or 0)
        misses = int(self.redis.get("cache:misses") or 0)
        total = hits + misses
        
        return {
            "hits": hits,
            "misses": misses,
            "hit_rate": f"{(hits/total*100):.1f}%" if total > 0 else "0%",
            "size": self.redis.dbsize()
        }
    
    def clear(self):
        \"\"\"Clear all cache entries.\"\"\" 
        # Clear only LLM cache keys
        for key in self.redis.scan_iter("llm_cache:*"):
            self.redis.delete(key)

# Usage
cache = RedisCache(ttl=3600)

def get_completion_cached(prompt: str, model: str = "gpt-3.5-turbo") -> str:
    # Check cache
    cached = cache.get(prompt, model)
    if cached:
        return cached
    
    # Call API
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = response.choices[0].message.content
    
    # Cache response
    cache.set(prompt, model, result)
    
    return result

✅ Benefits of Redis:
  - Persistent across restarts
  - Shared across multiple servers
  - Automatic TTL expiration
  - High performance
  - Built-in statistics
""")

## 🔄 Cache Invalidation Strategies

In [ ]:
print("""
# Cache Invalidation Strategies

## 1. Time-Based (TTL)

# Short TTL for dynamic content
cache.set(prompt, response, ttl=300)  # 5 minutes

# Long TTL for static content
cache.set(prompt, response, ttl=86400)  # 24 hours

## 2. Manual Invalidation

# Clear specific entry
cache.delete(prompt, model)

# Clear all entries
cache.clear()

## 3. Version-Based

def _make_key(prompt: str, model: str, version: str) -> str:
    data = json.dumps({
        "prompt": prompt,
        "model": model,
        "version": version  # Change version to invalidate all
    }, sort_keys=True)
    return hashlib.md5(data.encode()).hexdigest()

# Bump version to invalidate all caches
CACHE_VERSION = "v2"  # Was v1, now v2

## 4. Pattern-Based

# Invalidate by pattern
for key in redis.scan_iter("llm_cache:weather:*"):
    redis.delete(key)

## 5. Dependency-Based

class DependencyCache:
    def set_with_deps(self, key, value, dependencies):
        # Store value
        cache.set(key, value)
        
        # Track dependencies
        for dep in dependencies:
            redis.sadd(f"dep:{dep}", key)
    
    def invalidate_dependency(self, dep):
        # Get all keys dependent on this
        keys = redis.smembers(f"dep:{dep}")
        
        # Delete them all
        for key in keys:
            cache.delete(key)

# Example: Invalidate all user-related caches
dependency_cache.invalidate_dependency("user:123")

## 6. LRU (Least Recently Used)

# Configure Redis with maxmemory and LRU policy
# redis.conf:
maxmemory 1gb
maxmemory-policy allkeys-lru

# Redis automatically evicts least recently used entries
""")

print("\n💡 Best Practices:")
print("  1. Use TTL for most cases")
print("  2. Version-based for major updates")
print("  3. Manual invalidation for critical updates")
print("  4. LRU for memory-constrained systems")

## 📊 Cache Analytics

In [ ]:
class CacheAnalytics:
    """Track cache performance."""
    
    def __init__(self):
        self.hits = 0
        self.misses = 0
        self.total_latency_saved = 0.0
        self.total_cost_saved = 0.0
    
    def record_hit(self, latency_saved: float, cost_saved: float):
        """Record cache hit."""
        self.hits += 1
        self.total_latency_saved += latency_saved
        self.total_cost_saved += cost_saved
    
    def record_miss(self):
        """Record cache miss."""
        self.misses += 1
    
    def get_report(self) -> str:
        """Generate analytics report."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        
        report = f"""
📊 Cache Analytics Report
{'='*50}

Performance:
  Total requests: {total:,}
  Cache hits: {self.hits:,}
  Cache misses: {self.misses:,}
  Hit rate: {hit_rate:.1f}%

Savings:
  Latency saved: {self.total_latency_saved:.1f}s
  Cost saved: ${self.total_cost_saved:.4f}

Impact:
  Avg latency saved per hit: {self.total_latency_saved/max(self.hits,1):.3f}s
  Avg cost saved per hit: ${self.total_cost_saved/max(self.hits,1):.6f}
        """
        
        return report

# Example usage
analytics = CacheAnalytics()

# Simulate some cache operations
analytics.record_miss()  # First request
analytics.record_hit(latency_saved=1.5, cost_saved=0.002)  # Cached!
analytics.record_hit(latency_saved=1.5, cost_saved=0.002)
analytics.record_hit(latency_saved=1.5, cost_saved=0.002)
analytics.record_miss()  # New question
analytics.record_hit(latency_saved=1.5, cost_saved=0.002)

print(analytics.get_report())

## ✅ Summary

### Caching Strategies:

**1. In-Memory Cache**
```python
# Simple but limited
cache = SimpleCache()
cached = cache.get(prompt, model)
if not cached:
    cached = call_api()
    cache.set(prompt, model, cached)

✅ Use for:
  - Single server
  - Development
  - Small datasets
```

**2. TTL Cache**
```python
# Automatic expiration
cache = TTLCache(ttl_seconds=3600)
cache.set(prompt, model, response)  # Expires in 1 hour

✅ Use for:
  - Time-sensitive data
  - News, weather
  - Most LLM responses
```

**3. Redis Cache**
```python
# Production-ready
cache = RedisCache(ttl=3600)
cache.set(prompt, model, response)

✅ Use for:
  - Multiple servers
  - Production
  - High traffic
```

### TTL Recommendations:

| Content Type | TTL | Reasoning |
|--------------|-----|----------|
| Static FAQs | 24h | Rarely changes |
| User queries | 1h | Balance cost/freshness |
| News/weather | 5-15min | Frequently updates |
| Expensive ops | 6-12h | Maximize savings |
| Real-time | Don't cache | Always fresh |

### Best Practices:

**1. Always Use TTL**
```python
# ❌ BAD - No expiration
cache.set(key, value)

# ✅ GOOD - Auto expires
cache.set(key, value, ttl=3600)
```

**2. Track Metrics**
```python
if cached:
    metrics.record_hit()
else:
    metrics.record_miss()
```

**3. Hash Keys Properly**
```python
# Include all relevant parameters
key_data = {
    "prompt": prompt,
    "model": model,
    "temperature": temperature,  # Important!
    "version": CACHE_VERSION
}
key = hashlib.md5(json.dumps(key_data, sort_keys=True).encode()).hexdigest()
```

**4. Handle Cache Failures**
```python
try:
    cached = cache.get(prompt)
except Exception as e:
    logger.warning(f"Cache error: {e}")
    cached = None  # Fall back to API call
```

**5. Warm Up Cache**
```python
# Pre-populate common queries
common_queries = ["What is AI?", "How does ML work?"]
for query in common_queries:
    if not cache.get(query):
        response = call_api(query)
        cache.set(query, response)
```

### Cost Savings Example:

```
Without caching:
  100,000 requests × $0.002 = $200/day

With 70% hit rate:
  30,000 API calls × $0.002 = $60/day
  
Savings: $140/day = $4,200/month
```

### Next: `09_caching/02_semantic_caching.ipynb`